# Modelo para predecir si se han abierto las ventanas en la siguiente hora

In [21]:
from tensorflow.keras.models import load_model

model_temperatura = load_model('model_predict_temperaturas.keras')


In [38]:
import pandas as pd
from datetime import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df = pd.read_csv('entrenamiento_neuronal_calefaccion.csv')

df.head()

# Si la suma de suma_ventanas_arriba y suma_ventanas_abajo y puerta de la siguiente fila es mayor a 15 minutos 
df["ventanas_abiertas_siguiente_hora"] = (
    (df["suma_ventanas_arriba"].shift(-1) + df["suma_ventanas_abajo"].shift(-1) + df["sensor_puerta_1 Puerta"].shift(-1)) > 1200
)

df['horas'] = pd.to_datetime(df['time']).dt.time
df['dia_semana'] = pd.to_datetime(df['time']).dt.weekday  # 0 = lunes, 6 = domingo

# Filtra entre las 7:00 y las 21:00 y solo días entre semana (lunes a viernes)
df = df[(df['horas'] >= time(7, 0)) & 
        (df['horas'] <= time(21, 0)) & 
        (df['dia_semana'] < 5)]  # 0-4 para lunes a viernes

def calefaccion(row):
    if 7 <= row['hora'] < 11 and row['temperatura_calefaccion_y'] < 22:
        return True
    elif 11 <= row['hora'] < 16 and row['temperatura_calefaccion_y'] < 21:
        return True
    elif 16 <= row['hora'] < 18 and row['temperatura_calefaccion_y'] < 21.6:
        return True
    elif 18 <= row['hora'] < 20.67 and row['temperatura_calefaccion_y'] < 22:
        return True
    else:
        return False

# Aplica la función para crear la columna 'calefaccion_encendida'
df['calefaccion_encendida'] = df.apply(calefaccion, axis=1)

# Filtramos cuando la calefacción está encendida
df = df[df['calefaccion_encendida'] == True]

# Muestra el resultado
df.drop(columns=['horas','dia_semana'], inplace=True)

# Eliminar las columnas para predecir la temperatura
columns_to_drop = [
    "time",
    "sensor.sensor_temperatura_2_humidity",
    "sensor.sensor_temperatura_2_pressure",
    "sensor.sensor_temperatura_2_temperature",
    "sensor.sensor_temperatura_3_humidity",
    "sensor.sensor_temperatura_3_pressure",
    "sensor.sensor_temperatura_3_temperature", 
    'ventanas_abiertas_siguiente_hora', 
    'calefaccion_encendida',
    "temperatura_calefaccion_y"
]

df["temperatura_predicha"] = model_temperatura.predict(df[['sensor.sensor_temperatura_1_humidity',
       'sensor.sensor_temperatura_1_pressure',
       'sensor.sensor_temperatura_1_temperature', 'sensor_puerta_1 Puerta',
       'suma_ventanas_arriba', 'suma_ventanas_abajo', 'azimuth_mean',
       'elevacion_sol', 'temperatura_exterior', 'porcentaje_nubes',
        'hora', 'part_of_day', 'mes', 'season',
       'hora_sin', 'hora_cos', 'dia_1', 'dia_2', 'dia_3', 'dia_4', 'dia_5',
       'dia_6', 'dia_1', 'dia_2', 'dia_3', 'dia_4', 'dia_5', 'dia_6']])

# Hacemos drop de hora despues porque se usa en la prediccion de la temperatura
df.drop(columns=['hora'], inplace=True) 

X_df = df.drop(columns=["time","ventanas_abiertas_siguiente_hora",'sensor.sensor_temperatura_2_humidity', 'sensor.sensor_temperatura_2_pressure', 'sensor.sensor_temperatura_2_temperature', 'sensor.sensor_temperatura_3_humidity', 'sensor.sensor_temperatura_3_pressure', 'sensor.sensor_temperatura_3_temperature'])
y_df = df["ventanas_abiertas_siguiente_hora"]

X_df

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


,sensor.sensor_temperatura_1_humidity,sensor.sensor_temperatura_1_pressure,sensor.sensor_temperatura_1_temperature,sensor_puerta_1 Puerta,suma_ventanas_arriba,suma_ventanas_abajo,azimuth_mean,elevacion_sol,temperatura_exterior,porcentaje_nubes,temperatura_calefaccion_y,part_of_day,mes,season,hora_sin,hora_cos,dia_1,dia_2,dia_3,dia_4,dia_5,dia_6,calefaccion_encendida,temperatura_predicha
10,51.23,1013.77,19.48,0.0,0.0,0.0,122.93,2.68,7.7,93.0,19.6,3,12,2,0.942261,-0.334880,False,True,False,False,False,False,True,3661.182861
11,56.99,1015.33,21.06,0.0,0.0,0.0,131.90,10.43,7.8,49.0,19.6,3,12,2,0.816970,-0.576680,False,True,False,False,False,False,True,3732.848877
12,58.04,1015.40,21.16,0.0,0.0,0.0,144.64,18.67,7.0,48.0,20.5,3,12,3,0.631088,-0.775711,False,True,False,False,False,False,True,3770.799316
13,52.73,1015.58,20.65,0.0,0.0,0.0,158.21,24.19,7.7,8.0,21.5,3,12,3,0.398401,-0.917211,False,True,False,False,False,False,True,3848.456299
19,43.32,1014.60,20.58,0.0,0.0,0.0,238.30,1.41,11.8,100.0,21.0,2,12,3,-0.942261,-0.334880,False,True,False,False,False,False,True,3766.640137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2490,45.53,1009.65,20.78,3600.0,0.0,0.0,254.80,20.16,16.8,20.0,20.9,2,3,3,-0.942261,-0.334880,False,False,False,False,False,False,True,11057.092773
2491,46.49,1010.50,20.92,3600.0,0.0,0.0,265.73,7.96,16.2,52.0,21.2,2,3,3,-0.997669,-0.068242,False,False,False,False,False,False,True,11000.124023
2492,47.17,1011.50,21.36,3600.0,0.0,0.0,273.44,-1.18,15.5,87.0,21.3,1,3,3,-0.979084,0.203456,False,False,False,False,False,False,True,10961.653320
2493,47.73,1012.20,21.40,2366.0,0.0,0.0,283.89,-13.52,14.7,56.0,21.5,1,3,3,-0.887885,0.460065,False,False,False,False,False,False,True,7728.996094


In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import make_classification
import pandas as pd
import numpy as np
import itertools

# Dividir en train, val, test
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y_df, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Parámetros para probar
hidden_layers_list = [[32], [64, 32], [128, 64, 32]]
activations = ['relu', 'tanh']
optimizers_list = [
    ('adam', 0.001),
    ('sgd', 0.01),
    ('rmsprop', 0.0005),
]

results = []

# Generar combinaciones
combinations = list(itertools.product(hidden_layers_list, activations, optimizers_list))

for i, (hidden_layers_config, activation, (opt_name, lr)) in enumerate(combinations, 1):
    print(f"\n🔧 Probando modelo {i}/{len(combinations)}: capas={hidden_layers_config}, activation={activation}, optimizer={opt_name}, lr={lr}")

    model = models.Sequential()
    model.add(layers.Input(shape=(X_df.shape[1],)))
    for units in hidden_layers_config:
        model.add(layers.De2nse(units, activation=activation))
    model.add(layers.Dense(1, activation='sigmoid'))

    # Elegir optimizador
    if opt_name == 'adam':
        opt = optimizers.Adam(learning_rate=lr)
    elif opt_name == 'sgd':
        opt = optimizers.SGD(learning_rate=lr)
    elif opt_name == 'rmsprop':
        opt = optimizers.RMSprop(learning_rate=lr)

    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0, validation_data=(X_val, y_val))

    # Evaluación
    y_pred_probs = model.predict(X_test).ravel()
    y_pred = (y_pred_probs > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        'layers': str(hidden_layers_config),
        'activation': activation,
        'optimizer': opt_name,
        'lr': lr,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1
    })

# Mostrar resultados ordenados por F1
df = pd.DataFrame(results)
df_sorted = df.sort_values(by='f1_score', ascending=False).reset_index(drop=True)
print("\nTop resultados por F1 score:")
print(df_sorted.head(10).to_string(index=False))



🔧 Probando modelo 1/18: capas=[32], activation=relu, optimizer=adam, lr=0.001
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

🔧 Probando modelo 2/18: capas=[32], activation=relu, optimizer=sgd, lr=0.01
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

🔧 Probando modelo 3/18: capas=[32], activation=relu, optimizer=rmsprop, lr=0.0005
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

🔧 Probando modelo 4/18: capas=[32], activation=tanh, optimizer=adam, lr=0.001
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

🔧 Probando modelo 5/18: capas=[32], activation=tanh, optimizer=sgd, lr=0.01
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

🔧 Probando modelo 6/18: capas=[32], activation=tanh, optimizer=rmsprop, lr=0.0005
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

🔧 Probando modelo 7/18: capas=[64, 32], activation=relu, optimizer=adam, lr=0.001
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 227ms/step

🔧 Probando modelo 8/18: capas=[64, 32], activation=relu, optimizer=sgd, lr=0.01
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step

🔧 Probando modelo 9/18: capas=[64, 32], activation=r

/home/pablo/miniconda3/envs/tf-wsl-ia/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

🔧 Probando modelo 16/18: capas=[128, 64, 32], activation=tanh, optimizer=adam, lr=0.001
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 431ms/step

🔧 Probando modelo 17/18: capas=[128, 64, 32], activation=tanh, optimizer=sgd, lr=0.01
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

🔧 Probando modelo 18/18: capas=[128, 64, 32], activation=tanh, optimizer=rmsprop, lr=0.0005
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

Top resultados por F1 score:
       layers activation optimizer     lr  accuracy  precision  recall  f1_score
     [64, 32]       tanh      adam 0.0010  0.988889   0.975610   1.000  0.987654
         [32]       tanh      adam 0.0010  0.988889   0.975610   1.000  0.987654
         [32]       tanh       sgd 0.0100  0.988889   0.975610   1.000  0.987654
         [32]       tanh   rmsprop 0.0005  0.988889   0.975610   1.000  0.987654
[128, 64, 32]       relu   rmsprop 0.0005  0.988889   0.975610   1.000  0.987654
     [64, 32]       tanh       sgd 0.0100  0.988889   0.975610

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Modelo secuencial
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_df.shape[1],)),  # reemplaza input_dim por el número de características
    Dense(32, activation='relu'),
    Dense(output_dim, activation='softmax')  # reemplaza output_dim según tu problema (e.g. 1 para regresión o n clases para clasificación)
])

# Compilar el modelo
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)
